# Intelligent Systems – Task 1
## Warehouse Delivery Robot – Search Algorithms

This notebook implements the requirements of the task using:

- **Uniform Cost Search (UCS)** – Uninformed Search
- **A\*** – Informed Search
- **Manhattan Distance** as the heuristic

The program runs both algorithms on **MAP_A, MAP_B, and MAP_C** and reports:

- Path
- Path length
- Total path cost
- Nodes expanded
- Maximum frontier size
- Runtime
- Path displayed on the map


In [ ]:
# ============================================================
# Imports
# ============================================================

import heapq
import time


## 1. Define the Maps

The map symbols are:

- `S` = Start
- `G` = Goal
- `.` = Normal floor, cost 1
- `~` = Rough floor, cost 3
- `#` = Wall


In [ ]:
# ============================================================
# MAPS
# ============================================================

MAP_A = [
    "S..#....",
    "##.#.##.",
    "....~~..",
    ".####.#.",
    "...~..#G"
]

MAP_B = [
    "S~~~~~~~G",
    ".#######.",
    ".........",
    ".#.#.#.#.",
    "........."
]

MAP_C = [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G"
]

# Change this variable if you want to test one map separately
GRID = MAP_A

print("Selected Map:")
for row in GRID:
    print(row)


## 2. Find Start and Goal

In [ ]:
def find_start_goal(grid):
    start = None
    goal = None

    for row in range(len(grid)):
        for col in range(len(grid[row])):
            if grid[row][col] == 'S':
                start = (row, col)
            elif grid[row][col] == 'G':
                goal = (row, col)

    return start, goal


start, goal = find_start_goal(GRID)

print("Start:", start)
print("Goal:", goal)


## 3. Get Valid Neighbors

In [ ]:
def get_neighbors(grid, current):
    row, col = current

    # Up, Down, Left, Right
    directions = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1)
    ]

    for dr, dc in directions:
        new_row = row + dr
        new_col = col + dc

        # Check boundaries
        if new_row < 0 or new_row >= len(grid):
            continue

        if new_col < 0 or new_col >= len(grid[0]):
            continue

        # Cannot enter a wall
        if grid[new_row][new_col] == '#':
            continue

        yield (new_row, new_col)


## 4. Movement Cost

In [ ]:
def get_cell_cost(grid, position):
    row, col = position

    # Rough floor
    if grid[row][col] == '~':
        return 3

    # Normal floor and Goal
    return 1


## 5. Reconstruct the Path

In [ ]:
def reconstruct_path(parent, start, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)

        if current == start:
            break

        current = parent.get(current)

    path.reverse()

    if len(path) == 0 or path[0] != start:
        return []

    return path


## 6. Uniform Cost Search (UCS)

UCS is the required **uninformed search** algorithm.

It expands the node with the lowest accumulated path cost:

**g(n)**


In [ ]:
def uniform_cost_search(grid):
    start, goal = find_start_goal(grid)

    start_time = time.perf_counter()

    # Priority queue: (cost, counter, position)
    frontier = []
    counter = 0

    heapq.heappush(frontier, (0, counter, start))

    # Lowest known cost to each node
    cost_so_far = {start: 0}

    # Used to reconstruct the path
    parent = {start: None}

    nodes_expanded = 0
    max_frontier = 1

    while frontier:
        current_cost, _, current = heapq.heappop(frontier)

        # Ignore outdated entries
        if current_cost != cost_so_far.get(current):
            continue

        nodes_expanded += 1

        # Goal reached
        if current == goal:
            break

        for neighbor in get_neighbors(grid, current):
            movement_cost = get_cell_cost(grid, neighbor)
            new_cost = current_cost + movement_cost

            if new_cost < cost_so_far.get(neighbor, float('inf')):
                cost_so_far[neighbor] = new_cost
                parent[neighbor] = current

                counter += 1

                heapq.heappush(
                    frontier,
                    (new_cost, counter, neighbor)
                )

        max_frontier = max(max_frontier, len(frontier))

    runtime = time.perf_counter() - start_time

    if goal in cost_so_far:
        path = reconstruct_path(parent, start, goal)
        total_cost = cost_so_far[goal]
    else:
        path = []
        total_cost = None

    return {
        "algorithm": "Uniform Cost Search (UCS)",
        "path": path,
        "cost": total_cost,
        "nodes_expanded": nodes_expanded,
        "max_frontier": max_frontier,
        "runtime": runtime
    }


## 7. Manhattan Distance Heuristic

For A\*, we use:

**h(n) = |row1 - row2| + |col1 - col2|**

This estimates the number of grid moves from the current cell to the goal.


In [ ]:
def manhattan_distance(position, goal):
    row1, col1 = position
    row2, col2 = goal

    return abs(row1 - row2) + abs(col1 - col2)


## 8. A* Search

A\* is the required **informed search** algorithm.

It uses:

**f(n) = g(n) + h(n)**

where:

- `g(n)` = cost from the start to the current node
- `h(n)` = Manhattan distance estimate to the goal


In [ ]:
def a_star_search(grid):
    start, goal = find_start_goal(grid)

    start_time = time.perf_counter()

    # Priority queue: (f, g, counter, position)
    frontier = []
    counter = 0

    initial_h = manhattan_distance(start, goal)

    heapq.heappush(
        frontier,
        (initial_h, 0, counter, start)
    )

    cost_so_far = {start: 0}
    parent = {start: None}

    nodes_expanded = 0
    max_frontier = 1

    while frontier:
        f_cost, current_cost, _, current = heapq.heappop(frontier)

        # Ignore outdated entries
        if current_cost != cost_so_far.get(current):
            continue

        nodes_expanded += 1

        # Goal reached
        if current == goal:
            break

        for neighbor in get_neighbors(grid, current):
            movement_cost = get_cell_cost(grid, neighbor)
            new_cost = current_cost + movement_cost

            if new_cost < cost_so_far.get(neighbor, float('inf')):
                cost_so_far[neighbor] = new_cost
                parent[neighbor] = current

                h_cost = manhattan_distance(neighbor, goal)
                new_f = new_cost + h_cost

                counter += 1

                heapq.heappush(
                    frontier,
                    (new_f, new_cost, counter, neighbor)
                )

        max_frontier = max(max_frontier, len(frontier))

    runtime = time.perf_counter() - start_time

    if goal in cost_so_far:
        path = reconstruct_path(parent, start, goal)
        total_cost = cost_so_far[goal]
    else:
        path = []
        total_cost = None

    return {
        "algorithm": "A* Search",
        "path": path,
        "cost": total_cost,
        "nodes_expanded": nodes_expanded,
        "max_frontier": max_frontier,
        "runtime": runtime
    }


## 9. Display the Path on the Map

In [ ]:
def display_path(grid, path):
    display_grid = [list(row) for row in grid]

    for row, col in path:
        if display_grid[row][col] not in ['S', 'G']:
            display_grid[row][col] = '*'

    for row in display_grid:
        print("".join(row))


## 10. Print Results

In [ ]:
def print_result(result, grid):
    print("=" * 65)
    print("Algorithm:", result["algorithm"])
    print("=" * 65)

    if not result["path"]:
        print("No path found")
        print("Nodes expanded:", result["nodes_expanded"])
        print("Maximum frontier size:", result["max_frontier"])
        print(f"Runtime: {result['runtime']:.6f} seconds")
        return

    path_length = len(result["path"]) - 1

    print("Path:")
    print(result["path"])

    print("Path length:", path_length, "moves")
    print("Total path cost:", result["cost"])
    print("Nodes expanded:", result["nodes_expanded"])
    print("Maximum frontier size:", result["max_frontier"])
    print(f"Runtime: {result['runtime']:.6f} seconds")

    print("\nPath on map:")
    display_path(grid, result["path"])


## 11. Test Both Algorithms on the Selected Map

In [ ]:
ucs_selected = uniform_cost_search(GRID)
astar_selected = a_star_search(GRID)

print("\nUNIFORM COST SEARCH")
print_result(ucs_selected, GRID)

print("\nA* SEARCH")
print_result(astar_selected, GRID)


## 12. Run All Required Maps

The task requires both algorithms to be tested on:

- MAP_A
- MAP_B
- MAP_C


In [ ]:
def run_all_maps():
    maps = [
        ("MAP_A", MAP_A),
        ("MAP_B", MAP_B),
        ("MAP_C", MAP_C)
    ]

    all_results = {}

    for map_name, grid in maps:
        print("\n")
        print("#" * 70)
        print("MAP:", map_name)
        print("#" * 70)

        ucs_result = uniform_cost_search(grid)
        astar_result = a_star_search(grid)

        print("\n--- UCS ---")
        print_result(ucs_result, grid)

        print("\n--- A* ---")
        print_result(astar_result, grid)

        all_results[map_name] = {
            "UCS": ucs_result,
            "A*": astar_result
        }

    return all_results


all_results = run_all_maps()


## 13. Final Results Table

In [ ]:
def print_comparison_table(all_results):
    print("=" * 105)
    print("FINAL COMPARISON TABLE")
    print("=" * 105)

    print(
        f"{'Map':<10}"
        f"{'Algorithm':<28}"
        f"{'Path Length':<15}"
        f"{'Cost':<10}"
        f"{'Expanded':<12}"
        f"{'Max Frontier':<15}"
        f"{'Time':<12}"
    )

    print("-" * 105)

    for map_name in all_results:
        for algorithm_name in ["UCS", "A*"]:

            result = all_results[map_name][algorithm_name]

            if result["path"]:
                path_length = len(result["path"]) - 1
                cost = result["cost"]
            else:
                path_length = "No path"
                cost = "No path"

            print(
                f"{map_name:<10}"
                f"{result['algorithm']:<28}"
                f"{str(path_length):<15}"
                f"{str(cost):<10}"
                f"{str(result['nodes_expanded']):<12}"
                f"{str(result['max_frontier']):<15}"
                f"{result['runtime']:.6f}"
            )

    print("=" * 105)


print_comparison_table(all_results)


## 14. Algorithm Comparison

UCS is an uninformed search algorithm because it does not use information about the goal position. It selects the node with the lowest accumulated path cost.

A\* is an informed search algorithm because it combines the accumulated cost `g(n)` with the Manhattan heuristic `h(n)`.

For this warehouse problem, A\* is a suitable choice because it uses the goal direction to guide the search while considering the different movement costs. UCS is also optimal for the given positive movement costs.

**MAP_C** has no valid path, so both algorithms should report **No path found**.

**MAP_B** demonstrates why path length and path cost are different: the shortest route in number of moves is not necessarily the cheapest route because rough cells cost 3.


## 15. How to Change the Map

At the top of the notebook, change:

```python
GRID = MAP_A
```

to:

```python
GRID = MAP_B
```

or:

```python
GRID = MAP_C
```

The final section automatically tests all three required maps regardless of the selected `GRID`.
